In [1]:
!python -m pip install ipykernel numpy pillow ipywidgets ipyevents


[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Software simulation of mouse control
import numpy as np
from PIL import Image
from io import BytesIO
import ipywidgets as widgets
from IPython.display import display
from ipyevents import Event
import time

# Generating image, same as fixed-point python code in stage 
WIDTH, HEIGHT = 320, 240          # Lower resolution for faster rendering in software simulation
DISPLAY_WIDTH = 640
DISPLAY_HEIGHT = 480

MAX_ITER = 30
SCALE = 4096
FRAC_BITS = 12

RE_MIN, RE_MAX = -2.0, 2.0
IM_MIN, IM_MAX = -1.5, 1.5


DEFAULT_STEP = int(round((RE_MAX - RE_MIN) * SCALE / (WIDTH - 1)))
DEFAULT_ZR0 = int(round(RE_MIN * SCALE))
DEFAULT_ZI0 = int(round(IM_MIN * SCALE))


ROOTS_F = [
    (1.0, 0.0),
    (-0.5, 0.8660254),
    (-0.5, -0.8660254)
]

ROOTS = [
    (int(round(r * SCALE)), int(round(i * SCALE)))
    for (r, i) in ROOTS_F
]


COL = np.array([
    [230, 57, 70],
    [42, 157, 143],
    [69, 123, 157]
], dtype=int)


TOL = int(round(0.03 * SCALE))


DENOM_MIN = 1


def tdiv(a, b):
    """Truncate toward zero, like Verilog signed division."""
    if b == 0:
        return 0

    q = abs(a) // abs(b)

    if (a < 0) != (b < 0):
        return -q
    else:
        return q


def mul(a, b):
    """Q12 multiplication."""
    return tdiv(a * b, SCALE)


# Convert zoom/pan into register-style values
# Changing zoom/pan changes the register values
def view_to_registers(zoom_value=1.0, pan_x_value=0, pan_y_value=0):

    step = int(round(DEFAULT_STEP / zoom_value))


    if step < 1:
        step = 1


    centre_r = DEFAULT_ZR0 + (WIDTH // 2) * DEFAULT_STEP
    centre_i = DEFAULT_ZI0 + (HEIGHT // 2) * DEFAULT_STEP


    centre_r = centre_r + pan_x_value * step
    centre_i = centre_i + pan_y_value * step


    zr0 = centre_r - (WIDTH // 2) * step
    zi0 = centre_i - (HEIGHT // 2) * step

    return int(zr0), int(zi0), int(step)


# Generate one fixed-point fractal image
def generate_fixed_newton_image(zoom_value=1.0, pan_x_value=0, pan_y_value=0, max_iter=MAX_ITER):
    zr0, zi0, step = view_to_registers(zoom_value, pan_x_value, pan_y_value)

    img = np.zeros((HEIGHT, WIDTH, 3), dtype=np.uint8)

    for py in range(HEIGHT):
        for px in range(WIDTH):

            zr = zr0 + px * step
            zi = zi0 + py * step

            which = -1
            it = 0

            for it in range(max_iter):

                zr2 = mul(zr, zr) - mul(zi, zi)
                zi2 = tdiv(2 * zr * zi, SCALE)

                zr3 = mul(zr2, zr) - mul(zi2, zi)
                zi3 = mul(zr2, zi) + mul(zi2, zr)

                fr = zr3 - SCALE
                fi = zi3

                fpr = 3 * zr2
                fpi = 3 * zi2

                denom = mul(fpr, fpr) + mul(fpi, fpi)

                if denom <= DENOM_MIN:
                    break


                numr = mul(fr, fpr) + mul(fi, fpi)
                numi = mul(fi, fpr) - mul(fr, fpi)


                dr = tdiv(numr * SCALE, denom)
                di = tdiv(numi * SCALE, denom)


                zr = zr - dr
                zi = zi - di

                for k, (rr, ri) in enumerate(ROOTS):
                    if abs(zr - rr) < TOL and abs(zi - ri) < TOL:
                        which = k
                        break

                if which >= 0:
                    break

            if which < 0:
                img[py, px] = (0, 0, 0)
            else:

                shade256 = 256 - (it * 256) // max_iter

                if shade256 < 64:
                    shade256 = 64

                r = (int(COL[which][0]) * shade256) >> 8
                g = (int(COL[which][1]) * shade256) >> 8
                b = (int(COL[which][2]) * shade256) >> 8

                img[py, px] = (r, g, b)

    return img


# Convert NumPy image to PNG bytes
def image_to_png_bytes(img_array):
    img_array = np.asarray(img_array, dtype=np.uint8)
    image = Image.fromarray(img_array, mode="RGB")

    buffer = BytesIO()
    image.save(buffer, format="PNG")

    return buffer.getvalue()


# State variables
zoom = 1.0
pan_x = 0
pan_y = 0

dragging = False
drag_start_x = 0
drag_start_y = 0

last_render_time = 0


# Widgets

image_widget = widgets.Image(
    value=b"",
    format="png",
    width=DISPLAY_WIDTH,
    height=DISPLAY_HEIGHT
)

status = widgets.Label(value="Starting...")
reset_button = widgets.Button(description="Reset View")
reset_button.layout.width = "140px"



# Render function


def render():
    global last_render_time

    last_render_time = time.time()

    status.value = "Rendering..."

    img = generate_fixed_newton_image(
        zoom_value=zoom,
        pan_x_value=pan_x,
        pan_y_value=pan_y,
        max_iter=MAX_ITER
    )

    image_widget.value = image_to_png_bytes(img)

    zr0, zi0, step = view_to_registers(
        zoom_value=zoom,
        pan_x_value=pan_x,
        pan_y_value=pan_y
    )

    status.value = (
        f"Zoom = {zoom:.2f}, "
        f"Pan X = {pan_x}, "
        f"Pan Y = {pan_y}, "
        f"ZR0 = {zr0}, "
        f"ZI0 = {zi0}, "
        f"STEP = {step}"
    )



# Reset function

def reset_view(button=None):
    global zoom, pan_x, pan_y
    global dragging, drag_start_x, drag_start_y

    zoom = 1.0
    pan_x = 0
    pan_y = 0

    dragging = False
    drag_start_x = 0
    drag_start_y = 0

    status.value = "Resetting view..."
    render()


reset_button.on_click(reset_view)



# Mouse controls

def handle_mouse_event(event):
    global zoom, pan_x, pan_y
    global dragging, drag_start_x, drag_start_y

    event_type = event["type"]

    x = event.get("relativeX", 0)
    y = event.get("relativeY", 0)

    if event_type == "mousedown":
        dragging = True
        drag_start_x = x
        drag_start_y = y

        status.value = (
            f"Drag started at x={int(x)}, y={int(y)}. "
            f"Release mouse to apply pan."
        )

    elif event_type == "mousemove" and dragging:
        # Do not update the image while dragging.
        # Only show how far the drag is.
        dx = x - drag_start_x
        dy = y - drag_start_y

        dx_pixels = dx * WIDTH / DISPLAY_WIDTH
        dy_pixels = dy * HEIGHT / DISPLAY_HEIGHT

        status.value = (
            f"Dragging... release to apply | "
            f"dx = {int(round(dx_pixels))}, "
            f"dy = {int(round(dy_pixels))}"
        )

    elif event_type == "mouseup" and dragging:
        dragging = False

        dx = x - drag_start_x
        dy = y - drag_start_y

        dx_pixels = dx * WIDTH / DISPLAY_WIDTH
        dy_pixels = dy * HEIGHT / DISPLAY_HEIGHT

        # Apply pan only once after release
        pan_x -= int(round(dx_pixels))
        pan_y -= int(round(dy_pixels))

        render()

    elif event_type == "mouseleave":
        dragging = False
        status.value = (
            f"Drag cancelled | "
            f"Zoom = {zoom:.2f}, Pan X = {pan_x}, Pan Y = {pan_y}"
        )

    elif event_type == "wheel":
        delta_y = event.get("deltaY", 0)


        if delta_y < 0:
            zoom *= 1.2
        else:
            zoom /= 1.2

        # Limit zoom range
        if zoom < 0.2:
            zoom = 0.2

        if zoom > 50:
            zoom = 50

        render()


mouse_events = Event(
    source=image_widget,
    watched_events=[
        "mousedown",
        "mousemove",
        "mouseup",
        "mouseleave",
        "wheel"
    ]
)

mouse_events.prevent_default_action = True
mouse_events.on_dom_event(handle_mouse_event)


# Display UI

display(image_widget)
display(widgets.HBox([reset_button]))
display(status)

render()

Image(value=b'', height='480', width='640')

Label(value='Starting...')

C:\Users\weiki\AppData\Local\Temp\ipykernel_46344\3582105758.py:171: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  image = Image.fromarray(img_array, mode="RGB")


In [4]:
import sys
print(sys.executable)

c:\Users\yangl\AppData\Local\Python\pythoncore-3.14-64\python.exe
